In [55]:
import pandas as pd
import plotly.express as px

In [56]:
base = pd.read_csv("base_tratada.csv")

## Avaliando a influência sazonal da quantidade de pedidos

In [57]:
base_mes_ano = (
    base
    .groupby(["ano", "mes"])
    .agg(
        total_volumes=("total_volumes_mascarado", "sum"),
        qtd_setores=("cd_setor", "nunique")
    )
    .reset_index()
)

base_mes_ano["volume_medio_por_setor"] = (
    base_mes_ano["total_volumes"]
    / base_mes_ano["qtd_setores"]
)

base_mes_ano["mes_ano"] = (
    base_mes_ano["mes"].astype(str).str.zfill(2)
    + "/"
    + base_mes_ano["ano"].astype(str)
)

fig = px.bar(
    base_mes_ano,
    x="mes_ano",
    y="volume_medio_por_setor",
    title="Volume médio por setor por mês",
    labels={
        "mes_ano": "Mês/Ano",
        "volume_medio_por_setor": "Volume médio por setor"
    }
)

fig.show()

In [58]:
base_dia_semana = (
    base
    .groupby(["dia_semana_num", "dia_semana"], as_index=False)
    .agg(
        total_pedidos=("total_pedidos_mascarado", "sum")
    )
    .sort_values("dia_semana_num")
)

fig = px.bar(
    base_dia_semana,
    x="dia_semana",
    y="total_pedidos",
    title="Quantidade total de pedidos por dia da semana",
    labels={
        "dia_semana": "Dia da semana",
        "total_pedidos": "Quantidade de pedidos"
    },
    hover_data={
        "total_pedidos": ":,.0f",
        "dia_semana_num": False
    }
)

fig.show()

## Avaliando a variação da quantidade de pedidos dentro de um ciclo

In [59]:
base_dia_captacao = (
    base
    .groupby("dia_captacao", as_index=False)
    .agg(
        total_pedidos=("total_pedidos_mascarado", "sum")
    )
    .sort_values("dia_captacao")
)

fig = px.bar(
    base_dia_captacao,
    x="dia_captacao",
    y="total_pedidos",
    title="Quantidade total de pedidos por dia de captação",
    labels={
        "dia_captacao": "Dia de captação",
        "total_pedidos": "Quantidade de pedidos"
    },
    hover_data={
        "total_pedidos": ":,.0f"
    }
)

fig.show()

In [60]:
base_dia_ciclo = (
    base
    .groupby("dia_ciclo", as_index=False)
    .agg(
        total_pedidos=("total_volumes_mascarado", "sum")
    )
    .sort_values("dia_ciclo")
)

fig = px.bar(
    base_dia_ciclo,
    x="dia_ciclo",
    y="total_pedidos",
    title="Quantidade total de pedidos por dia do ciclo",
    labels={
        "dia_ciclo": "Dia do ciclo",
        "total_pedidos": "Quantidade de pedidos"
    },
    hover_data={
        "total_pedidos": ":,.0f"
    }
)

fig.show()

## Avaliando a distribuição do volume de pedidos por estado

In [61]:
base_estado = (
    base
    .groupby("estado", as_index=False)
    .agg(
        total_pedidos=("total_volumes_mascarado", "sum")
    )
    .sort_values("total_pedidos", ascending=False)
)

fig = px.bar(
    base_estado,
    x="estado",
    y="total_pedidos",
    title="Quantidade total de volumes por estado",
    labels={
        "estado": "Estado",
        "total_pedidos": "Quantidade de pedidos"
    },
    hover_data={
        "total_pedidos": ":,.0f"
    }
)

fig.show()

In [62]:
base_estado = (
    base
    .groupby("estado")
    .agg(
        total_volumes=("total_volumes_mascarado", "sum"),
        qtd_setores=("cd_setor", "nunique")
    )
    .reset_index()
)

base_estado["volume_medio_por_setor"] = (
    base_estado["total_volumes"]
    / base_estado["qtd_setores"]
)

base_estado = base_estado.sort_values(
    "volume_medio_por_setor",
    ascending=False
)

fig = px.bar(
    base_estado,
    x="estado",
    y="volume_medio_por_setor",
    title="Volume médio por setor por estado",
    labels={
        "estado": "Estado",
        "volume_medio_por_setor": "Volume médio por setor"
    },
    hover_data={
        "total_volumes": ":,.0f",
        "qtd_setores": True,
        "volume_medio_por_setor": ":,.1f"
    }
)

fig.show()

In [63]:
base_estado = base_estado.sort_values(
    "qtd_setores",
    ascending=False
)

fig = px.bar(
    base_estado,
    x="estado",
    y="qtd_setores",
    title="Quantidade de setores por estado",
    labels={
        "estado": "Estado",
        "qtd_setores": "Quantidade de setores"
    },
    hover_data={
        "qtd_setores": True
    }
)

fig.show()

## Avaliando se houve grande variação de setores ativos ao longo do período

In [64]:
x = 200

setor_mes = (
    base
    .groupby(["ano", "mes", "cd_setor"], as_index=False)
    .agg(
        total_volumes=("total_volumes_mascarado", "sum")
    )
)

setor_mes_filtrado = setor_mes[
    setor_mes["total_volumes"] > x
]

base_setores_mes = (
    setor_mes_filtrado
    .groupby(["ano", "mes"])
    .agg(
        qtd_setores=("cd_setor", "nunique")
    )
    .reset_index()
)

base_setores_mes = base_setores_mes.sort_values(["ano", "mes"])

base_setores_mes["mes_ano"] = (
    base_setores_mes["mes"].astype(str).str.zfill(2)
    + "/"
    + base_setores_mes["ano"].astype(str)
)

fig = px.bar(
    base_setores_mes,
    x="mes_ano",
    y="qtd_setores",
    title=f"Quantidade de setores com volume mensal maior que {x}",
    labels={
        "mes_ano": "Mês/Ano",
        "qtd_setores": "Quantidade de setores"
    }
)

fig.show()